# FedFlower Phase 3 — Evaluation & Grad-CAM

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

## Cell 1 — Install Grad-CAM & Imports

In [ ]:
!pip install grad-cam seaborn -q
import torch, torch.nn as nn, torchvision.models as models
import torchvision.datasets as datasets, torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix
import seaborn as sns
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Ready on {device}')

## Cell 2 — Upload best_model.pth

When the file picker appears, select `best_model.pth` from your laptop.

In [ ]:
from google.colab import files
import os
print('Upload best_model.pth from your laptop...')
uploaded = files.upload()
assert 'best_model.pth' in uploaded, '❌ Wrong filename — must be best_model.pth'
print(f'✅ Uploaded best_model.pth ({os.path.getsize("best_model.pth")/1e6:.1f} MB)')

## Cell 3 — Load Model & Dataset

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_f, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(512, num_classes))
    def forward(self, x): return self.backbone(x)

# Resize(256) → CenterCrop(224) matches the preprocessing used at inference time
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_data   = datasets.Flowers102('./data', split='test', download=True, transform=test_transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

model = FlowerCNN(num_classes=102).to(device)
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()
print(f'✅ Model and dataset loaded | Test images: {len(test_data)}')

## Cell 4 — Full Test Evaluation (Top-1, Top-5, Macro F1)

In [ ]:
all_preds, all_labels = [], []
top5_correct = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds  = outputs.max(1)
        _, top5   = outputs.topk(5, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        for i, lbl in enumerate(labels):
            if lbl.item() in top5[i].tolist():
                top5_correct += 1

all_preds, all_labels = np.array(all_preds), np.array(all_labels)
top1 = 100.0 * np.sum(all_preds == all_labels) / len(all_labels)
top5 = 100.0 * top5_correct / len(all_labels)
f1   = f1_score(all_labels, all_preds, average='macro')

print('=' * 45)
print('CENTRALIZED CNN — FINAL TEST RESULTS')
print('=' * 45)
print(f'Top-1 Accuracy : {top1:.2f}%')
print(f'Top-5 Accuracy : {top5:.2f}%')
print(f'Macro F1-Score : {f1:.4f}')
print(f'Test images    : {len(all_labels):,}')
print('=' * 45)

## Cell 5 — Confusion Matrix (Full 102×102)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)  # full 102×102
plt.figure(figsize=(20, 18))
sns.heatmap(cm, cmap='Blues', linewidths=0,
            xticklabels=range(1, 103), yticklabels=range(1, 103),
            cbar_kws={'shrink': 0.6})
plt.title('Confusion Matrix — All 102 Flower Classes', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.xticks(fontsize=5, rotation=90)
plt.yticks(fontsize=5, rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved confusion_matrix.png (102×102)')

## Cell 6 — Grad-CAM Visualizations (5 sample images)

Red = model focused here. Blue = ignored. A correct model highlights petals and stamens, not backgrounds.

In [ ]:
target_layers = [model.backbone.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

def denorm(t):
    img = t.numpy().transpose(1, 2, 0)
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    return np.clip(img, 0, 1).astype(np.float32)

NUM = 5
fig, axes = plt.subplots(NUM, 3, figsize=(12, NUM * 3))
fig.suptitle('Grad-CAM: What the Model Focuses On', fontsize=15, fontweight='bold')
sample_idx = [i * (len(test_data) // NUM) for i in range(NUM)]

for row, idx in enumerate(sample_idx):
    img_t, true_lbl = test_data[idx]
    inp = img_t.unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)
        pred   = logits.argmax(1).item()
        conf   = torch.softmax(logits, 1)[0, pred].item() * 100
    gc      = cam(input_tensor=inp, targets=[ClassifierOutputTarget(pred)])[0]
    img_np  = denorm(img_t)
    overlay = show_cam_on_image(img_np, gc, use_rgb=True)

    axes[row, 0].imshow(img_np);    axes[row, 0].set_title(f'True: Class {true_lbl+1}', fontsize=9);   axes[row, 0].axis('off')
    axes[row, 1].imshow(gc, cmap='jet'); axes[row, 1].set_title('Attention Heatmap', fontsize=9); axes[row, 1].axis('off')
    ok = '✓' if pred == true_lbl else '✗'
    axes[row, 2].imshow(overlay); axes[row, 2].set_title(f'Pred: Class {pred+1} ({conf:.0f}%) {ok}', fontsize=9); axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved gradcam_results.png')

## Cell 7 — Per-Class Accuracy Bar Chart

In [ ]:
pc_correct = np.zeros(102); pc_total = np.zeros(102)
for p, l in zip(all_preds, all_labels):
    pc_total[l] += 1
    if p == l: pc_correct[l] += 1
pc_acc = np.where(pc_total > 0, pc_correct / pc_total * 100, 0)

plt.figure(figsize=(18, 5))
colors = ['#1F4E79' if a >= 80 else '#E63946' if a < 60 else '#F4A460' for a in pc_acc]
plt.bar(range(1, 103), pc_acc, color=colors)
plt.axhline(y=pc_acc.mean(), color='black', linestyle='--', label=f'Mean: {pc_acc.mean():.1f}%')
plt.xlabel('Flower Class'); plt.ylabel('Accuracy (%)')
plt.title('Per-Class Accuracy — All 102 Flower Species', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150)
plt.show()

easy = np.argsort(pc_acc)[-5:][::-1]
hard = np.argsort(pc_acc)[:5]
print('Easiest classes:', [f'Class {c+1}: {pc_acc[c]:.0f}%' for c in easy])
print('Hardest classes:', [f'Class {c+1}: {pc_acc[c]:.0f}%' for c in hard])
print('✅ Saved per_class_accuracy.png')

## Cell 8 — Download All PNGs ⬇️

In [ ]:
from google.colab import files
print('Downloading evaluation outputs...')
files.download('confusion_matrix.png')
files.download('per_class_accuracy.png')
files.download('gradcam_results.png')
print('✅ All three PNGs downloaded')